# EVAL — tuned Qwen3-4B (non-hardcoded QLoRA SFT), harness-scored

The base-vs-tuned headline for the SFT model. The tuned model is fed the **same size-specific contract prompt it was trained on** (`user_contract(N)` — *"Write Python code to generate a NxN, fixed-grid, American-style crossword…"*), and every emitted `generate_crossword` program is scored through the **identical sandbox + scorer + real-dictionary check** as Claude (`pipeline.eval_opus_fleet.score_one`) on the **purified palette** (`WORD_LIST_FULLY_PURIFIED`, 24,542 words).

**Matched Claude baseline — EVAL 3** (identical size-specific prompt + purified palette): **0 / 150 valid** (GAP_ANALYSIS; unaugmented Opus times out on the full palette). So any non-zero here is the tuned model clearing an unaugmented-Opus floor of **0%**.

Sizes **7/9/11** (the trained sizes; 15 excluded, as in training). Every record of a size shares the identical prompt, so we build it directly per size and sample fresh programs at temperature 1.0 (the tuned analog of EVAL 3). **Runtime:** GPU, L4 (24 GB) / A100 (40 GB). **Order:** run top to bottom.

## 1. Get the code
Clone the repo (set your URL) or upload the folder and set `PROJECT_DIR`. The clone contains the word lists in `data/wordlists/` (incl. `WORD_LIST_FULLY_PURIFIED.txt`) and the `pipeline/` + `harness/` scoring code. **Push your latest local changes first** — the eval reflects whatever is committed here.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM.git"
import os
!git clone -q $REPO_URL slm || echo "clone skipped/failed — upload the folder instead"
PROJECT_DIR = "/content/slm"   # adjust if you uploaded elsewhere
assert os.path.isdir(os.path.join(PROJECT_DIR, "pipeline")), "Set PROJECT_DIR to the repo root"
%cd $PROJECT_DIR

## 2. GPU + install deps
Colab ships torch. We add `transformers`/`accelerate` (pinned to the training snapshot) to load the merged model, and `wordfreq` (only needed if you switch to the clean English palette). No `bitsandbytes` — the merged model loads in 16-bit directly.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU (L4/A100 recommended)"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q "transformers==4.53.*" "accelerate==1.8.*" wordfreq

> **Expected pip warning — safe to ignore.** Colab's pre-installed `gradio` wants a newer `huggingface-hub` than `transformers 4.53` pins. `gradio` is unused here; **do not upgrade `huggingface-hub`** (it would break `transformers`).

## 3. Point to the merged tuned model (on Drive)
Mount Drive and set `MODEL_DIR` to your **merged** model folder (`…-merged`) — the full standalone model, not the LoRA adapter. The training notebook (`colab_train_qlora.ipynb`, cell 9) saves it under `MyDrive/slm_ckpt/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# EDIT to your merged-model path on Drive (training notebook saves here):
MODEL_DIR = "/content/drive/MyDrive/slm_ckpt/qwen3-4b-crossword-qlora-merged"
import os
assert os.path.isdir(MODEL_DIR), f"MODEL_DIR not found: {MODEL_DIR}"
assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), \
    "not a full model dir (need config.json + model-*.safetensors, i.e. the -merged folder, not the adapter)"
print("model dir OK:", MODEL_DIR)

## 4. Load the tuned model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # left-pad so batched generation aligns at the prompt end
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
model.eval()
print("loaded:", model.config.model_type, "| dtype", next(model.parameters()).dtype, "| device", model.device)

## 5. Generation settings + batched helper
`GEN_TEMP = 1.0` matches Claude EVAL 3 (use 0.0 for greedy/deterministic — the model's single best output). `MAX_NEW_TOKENS = 8192` because a non-hardcoded 11×11 program embeds its grid template (~7k tokens) — **do not lower it below ~7000** or those programs truncate. KV cache grows with `BATCH × MAX_NEW_TOKENS`, so drop `BATCH` on a smaller GPU.

In [ ]:
GEN_TEMP       = 1.0     # match Claude EVAL 3; use 0.0 for greedy/deterministic
MAX_NEW_TOKENS = 8192    # 11x11 programs embed the template (~7k tok); do NOT lower below ~7000
BATCH          = 4       # KV cache ~ BATCH x MAX_NEW_TOKENS; lower to 1-2 on a 24 GB GPU

import torch
@torch.no_grad()
def generate_batch(pairs):
    """pairs: list of (system, user) -> list of completion strings (assistant turn only)."""
    outs = []
    for i in range(0, len(pairs), BATCH):
        chunk = pairs[i:i + BATCH]
        texts = [tok.apply_chat_template(
                    [{"role": "system", "content": s}, {"role": "user", "content": u}],
                    tokenize=False, add_generation_prompt=True)
                 for (s, u) in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=GEN_TEMP > 0, temperature=max(GEN_TEMP, 1e-5),
                             top_p=0.95, pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(new, skip_special_tokens=True))
        print(f"  generated {min(i + BATCH, len(pairs))}/{len(pairs)}", flush=True)
    return outs

## 6. Generate on the size-specific prompt, score through the harness
For each size in **7/9/11**, build the exact prompt the model was trained on (`SYSTEM` + `user_contract(size)`) and sample `PER_SIZE` fresh programs at `GEN_TEMP`. Each is scored by `score_one` on the purified palette with a real-dictionary check on every entry — identical to Claude EVAL 3. `GEN_TIMEOUT = 60 s` gives each emitted program the same execution budget as EVAL 3 (so `valid%` credits slow-but-correct grids).

In [ ]:
import os, json, time
# eval_opus_fleet reads ANTHROPIC_* at import; set dummies so nothing complains (no API calls here)
os.environ.setdefault("ANTHROPIC_BASE_URL", ""); os.environ.setdefault("ANTHROPIC_AUTH_TOKEN", "")
import pipeline.eval_opus_fleet as F
from pipeline.eval_opus_fleet import score_one, agg, table, purified_palette
from pipeline.contract_prompt import SYSTEM, user_contract
from pipeline.eval_harness import extract_code

SIZES    = [7, 9, 11]    # trained sizes (15 excluded, as in training)
PER_SIZE = 25            # samples per size at GEN_TEMP; 25 x 3 = 75 total
F.GEN_TIMEOUT = 60       # match EVAL 3: up to 60 s execution per emitted program
pal = purified_palette() # WORD_LIST_FULLY_PURIFIED (24,542 words) -- the EVAL 3 condition

# every record of a size shares the identical size-specific prompt, so build it directly + sample fresh
prompts = [(SYSTEM, user_contract(s), s) for s in SIZES for _ in range(PER_SIZE)]
print(f"{len(prompts)} prompts ({PER_SIZE}/size {SIZES})")
print("  example user prompt:", prompts[0][1].splitlines()[0])

print("generating...", flush=True)
comps = generate_batch([(s, u) for (s, u, sz) in prompts])

print("scoring on purified palette...", flush=True)
rows = []
for (s, u, sz), txt in zip(prompts, comps):
    code = extract_code(txt)
    if not code:
        rec = {"valid": 0, "fully": 0, "within": 0, "dict_frac": 0.0, "coverage": 0.0,
               "crossings": 0, "entries": 0, "filler": 0.0, "parsed": 0}
    else:
        rec = score_one(code, pal, sz, "vocabulary"); rec["parsed"] = 1
    rec["size"] = sz; rows.append(rec)

parse_rate = sum(r["parsed"] for r in rows) / len(rows)
print(f"\nparse rate (emitted a code block): {parse_rate*100:.0f}%")
ov = table(f"TUNED Qwen3-4B (non-hardcoded SFT), size-specific prompt, purified "
           f"(n={len(rows)}, temp={GEN_TEMP})", rows, SIZES)

## 7. Save results + base-vs-tuned comparison

In [ ]:
import os, json, time
os.makedirs("runs/eval", exist_ok=True)
out = f"runs/eval/tuned_nonhardcoded_{int(time.time())}.json"
summary = {"model": "qwen3-4b-crossword-qlora-merged (non-hardcoded SFT)",
           "condition": "size-specific contract prompt, purified palette",
           "n": len(rows), "parse_rate": parse_rate, "gen_temp": GEN_TEMP, "sizes": SIZES,
           "overall": ov, "by_size": {s: agg([r for r in rows if r["size"] == s]) for s in SIZES}}
json.dump(summary, open(out, "w", encoding="utf-8"), indent=2)
print("wrote", out)
!mkdir -p /content/drive/MyDrive/slm_runs/eval && cp "$out" /content/drive/MyDrive/slm_runs/eval/ 2>/dev/null || true

print("\n===== base vs tuned (size-specific prompt, purified palette) =====")
print(f"  Claude Opus 4.8 (EVAL 3, n=150): valid   0%   fullyOK   0%   within   0%   [GAP_ANALYSIS]")
print(f"  Tuned Qwen3-4B  (this run, n={len(rows)}): valid {ov['valid']*100:3.0f}%   "
      f"fullyOK {ov['fully']*100:3.0f}%   within {ov['within']*100:3.0f}%   (temp={GEN_TEMP})")

## 8. (Optional) dump the raw programs for inspection
Writes each emitted program to `runs/eval/tuned_nonhardcoded_progs/` so you can eyeball the actual generators the model produced.

In [ ]:
import os
os.makedirs("runs/eval/tuned_nonhardcoded_progs", exist_ok=True)
for k, ((s, u, sz), txt) in enumerate(zip(prompts, comps)):
    code = extract_code(txt) or txt
    open(f"runs/eval/tuned_nonhardcoded_progs/prog_{k:03d}_s{sz}.py", "w", encoding="utf-8").write(code)
print("saved", len(prompts), "programs under runs/eval/tuned_nonhardcoded_progs/")

## Next
If the tuned model clears a meaningful bar here (vs Claude's **0%** at EVAL 3), record the numbers in `GAP_ANALYSIS.md` as the **tuned column** of the base-vs-tuned story. To also test the clean English palette (Claude EVAL 2's condition), swap `purified_palette()` for `english_palette(max(SIZES))` from `pipeline.eval_selfmodel` (add `wordfreq`, already installed).